In [44]:
import pandas as pd

df = pd.read_csv('../dataset/regular_season_box_scores_2010_2024_part_1.csv')
df = df.drop(columns = ['jerseyNum', 'comment'], axis = 1)
df['position'] = df.groupby('personName')['position'].transform(lambda x: x.ffill().bfill())
df = df.dropna()
df.info()

C:\Users\sidne\AppData\Local\Temp\ipykernel_15104\2202791194.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['position'] = df.groupby('personName')['position'].transform(lambda x: x.ffill().bfill())


<class 'pandas.core.frame.DataFrame'>
Index: 112421 entries, 0 to 141492
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   season_year              112421 non-null  object 
 1   game_date                112421 non-null  object 
 2   gameId                   112421 non-null  int64  
 3   matchup                  112421 non-null  object 
 4   teamId                   112421 non-null  int64  
 5   teamCity                 112421 non-null  object 
 6   teamName                 112421 non-null  object 
 7   teamTricode              112421 non-null  object 
 8   teamSlug                 112421 non-null  object 
 9   personId                 112421 non-null  int64  
 10  personName               112421 non-null  object 
 11  position                 112421 non-null  object 
 12  minutes                  112421 non-null  object 
 13  fieldGoalsMade           112421 non-null  int64  
 14  fieldGoal

In [47]:
player_stats = df.drop(columns = ['minutes', 'season_year', 'game_date', 'gameId', 'matchup', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId'], axis = 1).copy()
player_positions = player_stats.groupby(by = 'personName').agg({'position': lambda x: ', '.join(x.unique())}).reset_index()
player_stats_group = player_stats.groupby(by = 'personName').sum().reset_index()
player_stats_group = player_stats_group.drop(columns = 'position', axis = 1)
#player_stats_group.sort_values(by = 'points', ascending = False)
player_stats = pd.merge(player_positions, player_stats_group, on = 'personName', how = 'outer')
player_stats['fieldGoalsPercentage'] = player_stats.apply(lambda x: (x['fieldGoalsMade'] / x['fieldGoalsAttempted']) * 100 if x['fieldGoalsAttempted'] != 0 else 0, axis = 1)
player_stats['threePointersPercentage'] = player_stats.apply(lambda x: (x['threePointersMade'] / x['threePointersAttempted']) * 100 if x['threePointersAttempted'] != 0 else 0, axis = 1)
player_stats['freeThrowsPercentage'] = player_stats.apply(lambda x: (x['freeThrowsMade'] / x['freeThrowsAttempted']) * 100 if x['freeThrowsAttempted'] != 0 else 0, axis = 1)

player_stats[player_stats['personName']  == 'Stephen Curry']

,personName,position,fieldGoalsMade,fieldGoalsAttempted,fieldGoalsPercentage,threePointersMade,threePointersAttempted,threePointersPercentage,freeThrowsMade,freeThrowsAttempted,...,reboundsOffensive,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints
608,Stephen Curry,G,7556,15955,47.358195,3581,8425,42.504451,3576,3925,...,581,3572,4153,5647,1321,216,2744,1963,22269,6065
